# N=48 leak-free, L=10: the depth ladder is NOT monotone, and the bracket does not narrow

**Job `50607549`, submitted 2026-10-31 22:46:36, started 22:48:55, ended 2026-11-01 06:47:46
(8 h 58 m 51 s wall, across the DST fall-back), COMPLETED 0:0 on `sh04-02n01`.
Commit at submit `44f581f`, seed 0, one change from `50527633`: `num_layers` 12 -> 10.**

Finding 38 bracketed L\*(48, leak-free, ff=1024) in **(8, 12]** at rho=0.60 and the obvious next
move was the midpoint. The previous notebook said so in as many words -- *"One more cell at L=10
halves the bracket; that is the cheapest next job."*

**It did not halve the bracket. L=10 came back BELOW L=8.**

On the shared n=1500 pool L=10 scores **0.8007**, against L=8's **0.8660** -- a paired difference of
**-0.0653**, discordant **92-190**, **McNemar exact p = 5.4e-09**. Two extra layers, the same data, the
same budget, the same instances, and the deeper model is resolvably *worse*. The ladder now reads

| L | 3 | 4 | 6 | 8 | **10** | 12 |
|---|---|---|---|---|---|---|
| pool | 0.0093 | 0.3153 | 0.6153 | 0.8660 | **0.8007** | 0.9780 |

which is **not monotone in depth**.

**What this does and does not establish.** The p-value compares two *checkpoints* on the same
instances: it establishes that these two trained models differ, and by how much. It does **not**
establish that retraining L=10 would land below L=8 again. There is **one seed per cell** on this
entire ladder, and no rung has ever been replicated. So the honest headline is a methodological one:

> **you cannot read a knee off a single-seed ladder that is not monotone**, and this ladder is not.

Finding 38's "(8, 12]" survives -- L=12 still clears 6/6 bins and L=8 still clears 3/6 -- but the
reason it cannot be narrowed by interpolation is now measured rather than assumed. Pre-registration
(a)'s own falsifier says a miss puts L\* in (10, 12]; that reading is only valid if depth helps
monotonically, which is exactly what this cell refutes.

**Two things argue the inversion is in the trained model, not in the eval.** Grammar is clean
(`well_formed_rate` >= 0.9905 in every bin, `truncated_rate` 0.0007, `outcome_accuracy` 0.9953), so
this is a deduction deficit and not finding 25's formatting collapse. And `val_nll` independently
ranks the four deep rungs in **exactly the pool's order, inversion included** (L=12 1.4993 <
L=8 1.5033 < L=10 1.5059 < L=6 1.5129). That is concordance, not vindication -- `val_nll` is
disqualified as a ranking statistic across models (findings 4/20/25, r=+0.33 over 34 runs) -- but it
does mean the deficit was already present at the token level before free generation amplified it.

In [ ]:
import glob, io, json, math, os, re
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt

S = os.environ["SCRATCH"] + "/spin_gnn"
DIRS = [f"{S}/sudoku_n48_leakfree_lstar_2026-10-29",   # 50099945: L=6, 4, 3
        f"{S}/sudoku_n48_leakfree_lstar_2026-10-31",   # 50527633: L=12, 50527634: L=8
        f"{S}/sudoku_n48_leakfree_L10_2026-10-31"]     # 50607549: L=10 + all five re-evaluated
# The L10 job stamped its output dir with its 10-31 START date, not the 11-01
# date it finished on, and named its bin CSV depth_L10_* rather than depth_lstar_*.
DEPTH_GLOBS = ["depth_lstar_*.csv", "depth_L10_*.csv"]
GATE, LADDER = 0.90, [12, 10, 8, 6, 4, 3]

lab = lambda c: "L=%d" % int(re.search(r"_L(\d+)_", c).group(1))
depth_of = lambda c: int(re.search(r"_L(\d+)_", c).group(1))

def wilson(k, n, z=1.96):
    p, d = k / n, 1 + z * z / n
    c = p + z * z / (2 * n)
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - h) / d, (c + h) / d)

def mcnemar_exact(b, c):
    '''Two-sided exact McNemar on discordant counts b and c.'''
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)

def ca_trend(ks, ns, xs):
    '''Cochran-Armitage trend z over ordered bins. Negative = falling with x.'''
    ks, ns, xs = np.asarray(ks, float), np.asarray(ns, float), np.asarray(xs, float)
    N, K = ns.sum(), ks.sum()
    pbar, xbar = K / N, (ns * xs).sum() / N
    var = pbar * (1 - pbar) * (ns * (xs - xbar) ** 2).sum()
    return (ks * (xs - xbar)).sum() / np.sqrt(var) if var > 0 else float("nan")

pool_csvs, bin_csvs, npzs = [], [], []
for d in DIRS:
    pool_csvs += sorted(glob.glob(f"{d}/sharedpool_n1500_*.csv"))
    npzs += sorted(glob.glob(f"{d}/sharedpool_n1500_*_per_instance.npz"))
    for pat in DEPTH_GLOBS:
        bin_csvs += sorted(glob.glob(f"{d}/{pat}"))
print(f"{len(pool_csvs)} pool CSVs, {len(bin_csvs)} depth CSVs, {len(npzs)} per-instance npz")

## 1. Provenance, config and seed

The claim "only `num_layers` differs" is the load-bearing one for the whole ladder, so it is checked
against each checkpoint's own `config.json` rather than against the sbatch scripts.

In [ ]:
CELLDIRS = {12: f"{S}/sudoku_n48_leakfree_lstar_2026-10-31/L12",
            10: f"{S}/sudoku_n48_leakfree_L10_2026-10-31/L10",
             8: f"{S}/sudoku_n48_leakfree_lstar_2026-10-31/L08",
             6: f"{S}/sudoku_n48_leakfree_lstar_2026-10-29/L06"}
KEYS = ("d_model", "nhead", "dim_feedforward", "n_train_instances", "epochs", "lr",
        "lr_schedule", "warmup_epochs", "batch_size", "shuffle_adj", "shuffle_events",
        "seed", "rho", "rho_max", "n_nodes", "degree", "n_colors", "norm_first", "max_len")

rows, base = [], None
for L in sorted(CELLDIRS, reverse=True):
    d = f"{CELLDIRS[L]}/d0256_h04_L{L:02d}_ff01024_lr1e-03_pre_shuf"
    cfg = json.load(open(f"{d}/config.json"))
    hist = json.load(open(f"{d}/history.json"))
    v = [e["nll"] for e in hist]; a = [e["accuracy"] for e in hist]
    npar = int(pd.read_csv(f"{CELLDIRS[L]}/sweep.csv").n_params.iloc[0])
    rows.append(dict(L=L, n_params_M=npar / 1e6, epochs=len(v),
                     final_val_nll=v[-1], min_val_nll=min(v), final_tok_acc=a[-1]))
    cur = {k: cfg[k] for k in KEYS}
    if base is None:
        base = cur
    else:
        delta = {k: (base[k], cur[k]) for k in KEYS if base[k] != cur[k]}
        assert not delta, f"L={L} differs from L=12 beyond num_layers: {delta}"
print("config check: num_layers is the ONLY difference across L=12/10/8/6")
print(f"seed {base['seed']}, n_train {base['n_train_instances']}, epochs {base['epochs']}, "
      f"ff {base['dim_feedforward']}, shuffle_adj {base['shuffle_adj']}")
prov = pd.DataFrame(rows)
print()
print(prov.round(4).to_string(index=False))

## 2. The metric of record: the shared pool, n=1500

`sudoku_agg_eval`, `seed_start=5e10`, disjoint from every training set, instance *i* at `seed_start+i`.
Each of the three jobs re-evaluated the earlier cells, so several checkpoints were scored two or three
times. Those repeats are asserted to agree **instance for instance** before anything is pooled -- that
assertion is pre-registration (d), and it is also the only determinism check this eval path gets.

In [ ]:
solved, seen_in, seed_starts = {}, {}, set()
for p in npzs:
    job = re.search(r"_(\d+)_per_instance", p).group(1)
    z = np.load(p, allow_pickle=True)
    seed_starts.add(int(z["seed_start"]))
    for i, name in enumerate(z["names"]):
        key, v = lab(str(name)), np.asarray(z["solved"][i], bool)
        if key in solved:
            agree = bool((solved[key] == v).all())
            print(f"  repeat eval of {key:<5}: job {job} vs {seen_in[key]} -> "
                  f"{'IDENTICAL on all %d' % v.size if agree else 'DISAGREE'}")
            assert agree, f"{key} disagrees between {job} and {seen_in[key]}"
        else:
            solved[key], seen_in[key] = v, job
assert len(seed_starts) == 1, f"pools differ: {seed_starts}"
print(f"\nseed_start {seed_starts.pop()}; cells: {[k for k in ('L=%d' % L for L in LADDER)]}")

tab = pd.DataFrame([dict(cell="L=%d" % L, L=L, n=solved["L=%d" % L].size,
                         solve_rate=solved["L=%d" % L].mean(),
                         wilson_lo=wilson(int(solved["L=%d" % L].sum()), solved["L=%d" % L].size)[0],
                         wilson_hi=wilson(int(solved["L=%d" % L].sum()), solved["L=%d" % L].size)[1])
                    for L in LADDER])
tab["clears_gate"] = (tab.solve_rate >= GATE) & (tab.wilson_lo >= 0.88)
print()
print(tab.round(4).to_string(index=False))

mono = bool((tab.sort_values("L").solve_rate.diff().dropna() > 0).all())
print(f"\nladder monotone increasing in depth? {mono}")

## 3. Every paired contrast on the same 1500 instances

The inversion is the `L=10 vs L=8` row. Note what the test is and is not: McNemar on paired instances
answers *"do these two checkpoints differ?"*, not *"would a retrained L=10 differ?"*.

In [ ]:
keys = ["L=%d" % L for L in LADDER]
con = []
for i, a in enumerate(keys):
    for b in keys[i + 1:]:
        x, y = solved[a], solved[b]
        won, lost = int((x & ~y).sum()), int((~x & y).sum())
        con.append(dict(a=a, b=b, delta=x.mean() - y.mean(), a_wins=won, b_wins=lost,
                        p=mcnemar_exact(won, lost)))
con = pd.DataFrame(con)
print(con.to_string(index=False, formatters={"delta": "{:+.4f}".format, "p": "{:.3g}".format}))
print()
r = con[(con.a == "L=10") & (con.b == "L=8")].iloc[0]
print(f"THE INVERSION: L=10 minus L=8 = {r.delta:+.4f}, discordant {r.a_wins}-{r.b_wins}, "
      f"p = {r.p:.3g}  ->  the DEEPER model is resolvably WORSE")

## 4. Depth bins, the gate, and the trend in T

Two densities. At rho=0.60 the deepest bin is T=8; at rho=0.75 it is T=4. The realized density is
reported because a bin that is deeper *and* sparser would confound "depth" with "density" -- it is
pinned to within 0.0005 here.

In [ ]:
bins = pd.concat([pd.read_csv(p) for p in bin_csvs], ignore_index=True)
bins["L"] = bins.checkpoint.map(depth_of)
bins = bins.drop_duplicates(subset=["checkpoint", "rho_requested", "T"])

summ = {}
for rho in sorted(bins.rho_requested.unique(), reverse=True):
    sub = bins[bins.rho_requested == rho]
    piv = sub.pivot_table(index="T", columns="L", values="solve_rate")
    piv = piv[sorted(piv.columns, reverse=True)]
    nn = sub.pivot_table(index="T", columns="L", values="n").max(axis=1).astype(int)
    out = piv.copy(); out.columns = ["L=%d" % c for c in out.columns]; out.insert(0, "n", nn)
    print(f"=== rho {rho:.2f}, realized {sub.realized_rho.min():.6f}-{sub.realized_rho.max():.6f}")
    print(out.round(4).to_string())
    nb = sub["T"].nunique()
    clears = {int(L): int(((sub.L == L) & (sub.solve_rate >= GATE)).sum())
              for L in sorted(sub.L.unique(), reverse=True)}
    lstar = [L for L in sorted(sub.L.unique()) if clears[L] == nb]
    print(f"  bins >= {GATE:.2f}, of {nb}: {clears}")
    print(f"  L* on this density: {min(lstar) if lstar else 'UNDEFINED'}")
    for L in sorted(sub.L.unique(), reverse=True):
        q = sub[sub.L == L].sort_values("T")
        print(f"    L={L:<3} {q.solve_rate.max():.3f} -> {q.solve_rate.min():.3f}, "
              f"Cochran-Armitage z = {ca_trend((q.solve_rate * q.n).round(), q.n, q['T']):+.2f}")
    summ[rho] = (piv, clears, nb)
    print()

piv60 = summ[0.60][0]
print("=== rho=0.60, L=10 minus L=8 per bin (negative = the deeper model is worse)")
print((piv60[10] - piv60[8]).round(4).to_string())
n60 = bins[(bins.rho_requested == 0.60) & (bins.L == 10)].set_index("T")["n"]
print("\nThe deficit is near zero in the shallowest bin (-0.009 at T=3) and five to ten times larger")
print("in every bin from T=4 on. It is NOT monotone in T -- T=7 is the mildest of those five -- but")
print(f"the three deepest bins hold only {', '.join(str(int(n60[t])) for t in (6, 7, 8))} instances,")
print("so their bin-to-bin wobble is not resolvable. The defensible statement is the weaker one:")
print("the two extra layers cost nothing on the easiest instances and cost 4-9 points on the rest.")

## 5. The format controls: is this deduction or grammar?

Finding 25 found a deep-thin cell at N=96 failing at *grammar* rather than deduction, which looked
like a capacity result until the well-formedness rate was divided out. The same check here.

In [ ]:
g = pd.concat([pd.read_csv(p) for p in pool_csvs], ignore_index=True)
g["cell"] = g.checkpoint.map(lab)
print("pool-level:")
print(g.groupby("cell")[["truncated_rate", "outcome_accuracy"]].max().round(4).to_string())
print("\nwell_formed_rate per bin at rho=0.60:")
s60 = bins[bins.rho_requested == 0.60]
wf = s60.pivot_table(index="T", columns="L", values="well_formed_rate")
print(wf[sorted(wf.columns, reverse=True)].round(4).to_string())
print(f"\nL=10 minimum well_formed_rate over all bins: {s60[s60.L == 10].well_formed_rate.min():.4f}")
print("Clean -> the L=10 deficit is deduction, not formatting. Finding 25's mechanism is excluded.")

## 6. Plots

In [ ]:
plt.rcParams.update({"font.size": 12})
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

t = tab.sort_values("L")
ax = axes[0]
ax.errorbar(t.L, t.solve_rate, yerr=[t.solve_rate - t.wilson_lo, t.wilson_hi - t.solve_rate],
            fmt="o-", lw=2.5, ms=9, capsize=5, color="#2b6cb0", rasterized=True)
ax.plot([10], [t[t.L == 10].solve_rate.iloc[0]], "o", ms=16, mfc="none", mec="#c53030", mew=3,
        rasterized=True)
ax.annotate("L=10 falls BELOW L=8\n-0.0653, p=5e-09", xy=(10, 0.80), xytext=(5.6, 0.50),
            fontsize=11, color="#c53030",
            arrowprops=dict(arrowstyle="->", color="#c53030", lw=1.8))
ax.axhline(GATE, color="k", ls="--", lw=2)
ax.text(3.1, GATE + 0.02, "0.90 gate", fontsize=11)
ax.set_xticks(t.L); ax.set_xlabel("num_layers")
ax.set_ylabel("shared-pool solve rate (n=1500)")
ax.set_ylim(-0.05, 1.08)
ax.set_title("The ladder is not monotone")

ax = axes[1]
s60 = bins[bins.rho_requested == 0.60]
for L in sorted(s60.L.unique(), reverse=True):
    q = s60[s60.L == L].sort_values("T")
    style = dict(lw=3.5, ms=10) if L in (8, 10) else dict(lw=2, ms=7, alpha=0.65)
    ax.plot(q["T"], q.solve_rate, "o-", label=f"L={L}", rasterized=True, **style)
ax.axhline(GATE, color="k", ls="--", lw=2)
ax.set_xticks(sorted(s60["T"].unique()))
ax.set_xlabel("T, DD rounds the instance needs")
ax.set_ylabel("solve rate")
ax.set_ylim(-0.05, 1.08)
ax.set_title("rho=0.60: L=10 (bold) sits under L=8 (bold) at every T")
ax.legend(loc="lower left", ncol=2, fontsize=10)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

ax = axes[0]
d = (piv60[10] - piv60[8])
ax.bar([str(i) for i in d.index], d.values,
       color=["#c53030" if v < 0 else "#2f855a" for v in d.values], rasterized=True)
ax.axhline(0, color="k", lw=1)
ax.set_xlabel("T"); ax.set_ylabel("solve rate, L=10 minus L=8")
ax.set_title("Near zero at T=3, 4-9 points everywhere else")

ax = axes[1]
p = prov.sort_values("L")
ax.plot(p.L, p.final_val_nll, "s-", lw=2.5, ms=9, color="#6b46c1", rasterized=True)
for _, r in p.iterrows():
    ax.annotate(f"pool {tab[tab.L == r.L].solve_rate.iloc[0]:.3f}", (r.L, r.final_val_nll),
                textcoords="offset points", xytext=(0, 9), ha="center", fontsize=10)
ax.set_xticks(p.L); ax.set_xlabel("num_layers"); ax.set_ylabel("final val_nll")
ax.set_title("val_nll reproduces the inversion (lower is better)")
fig.tight_layout()
plt.show()

## 7. Comparison to the previous best on the same eval

Same eval, same 1500 instances, same `seed_start`. The previous best on this ladder is finding 38's
L=12, and it is untouched -- this cell neither beats it nor approaches it.

In [ ]:
prev, new = solved["L=12"], solved["L=10"]
won, lost = int((new & ~prev).sum()), int((~new & prev).sum())
print(f"previous best on this eval   L=12  {prev.mean():.4f}  "
      f"[{wilson(int(prev.sum()), prev.size)[0]:.4f}, {wilson(int(prev.sum()), prev.size)[1]:.4f}]")
print(f"this run                     L=10  {new.mean():.4f}  "
      f"[{wilson(int(new.sum()), new.size)[0]:.4f}, {wilson(int(new.sum()), new.size)[1]:.4f}]")
print(f"paired delta {new.mean() - prev.mean():+.4f}, discordant {won}-{lost}, "
      f"p = {mcnemar_exact(won, lost):.3g}")
print("\nNO IMPROVEMENT. The best N=48 leak-free cell remains finding 38's L=12 at 0.9780.")
print("This run's value is the negative result, not a new number at the top of the table.")

## 8. Pre-registrations, scored

Quoted verbatim from the header of `scripts/slurm/sudoku2_n48_L10_narrow.sbatch`, committed as
`44f581f` **before** the job was submitted.

In [ ]:
s60 = bins[bins.rho_requested == 0.60]
c10 = s60[s60.L == 10].sort_values("T")
cleared = [int(r["T"]) for _, r in c10.iterrows() if r.solve_rate >= GATE]
missed = [int(r["T"]) for _, r in c10.iterrows() if r.solve_rate < GATE]
pool10, pool8 = solved["L=10"].mean(), solved["L=8"].mean()
contig = missed == list(range(min(missed), max(missed) + 1)) and max(missed) == int(c10["T"].max())
rr = s60.realized_rho
print(f"(a) HEADLINE: 'L=10 clears 0.90 in ALL SIX rho=0.60 bins' ->")
print(f"    clears {len(cleared)} of 6 (cleared T={cleared}, missed T={missed})  ** FALSIFIED **")
print(f"(b) POOL in [0.93, 0.97] and resolvably ABOVE L=8 ->")
print(f"    {pool10:.4f}; falsifier B was '<= 0.9220' -> ** FALSIFIED **, and the sign is reversed:")
print(f"    resolvably BELOW L=8 ({pool10 - pool8:+.4f}, p={con[(con.a=='L=10')&(con.b=='L=8')].p.iloc[0]:.3g})")
print(f"(c) CONDITIONAL (scored because (a) failed): missed bins are the DEEPEST and contiguous ->")
print(f"    missed T={missed}, contiguous run at the top: {contig}  ** CONFIRMED **")
print(f"(d) CONTROLS: realized density within 0.001 of 0.5836, repeats agree instance for instance ->")
print(f"    realized {rr.min():.6f}-{rr.max():.6f} (max deviation {max(abs(rr.min()-0.5836), abs(rr.max()-0.5836)):.6f});")
print(f"    all repeat evaluations IDENTICAL (asserted in section 2)  ** CONFIRMED **")
print("\n2 of 4. The two that failed are the two that predicted the result;")
print("the two that held are the conditional and the controls.")

## 9. What this means, and the hypothesis for the next run

**What is established.** Two checkpoints trained identically except for `num_layers` (10 vs 8), on the
same 240000 x 100ep budget, score 0.8007 and 0.8660 on the same 1500 held-out instances, p = 5.4e-09.
The deeper one is worse, the deficit is near zero only on the easiest bin, and grammar is clean. Config confounds are excluded: the only
difference between the two runs is the layer count, verified from each checkpoint's own `config.json`.

**What is not established.** Whether *depth 10* is bad, or whether *this particular run* is. One seed
per cell, no replicates anywhere on the ladder. Two readings survive:

- **H1 (leading): optimization noise.** A single unlucky initialization/trajectory at L=10. Nothing in
  the training curve flags it -- loss descends monotonically to epoch 100 and the final val_nll is only
  +0.0026 above L=8's -- but a 0.0026 nll difference mapping to a 0.065 solve-rate difference shows how
  little nll has to move to swing free generation, so a noise-sized perturbation is sufficient.
- **H2: a real non-monotonicity.** Some depths genuinely train worse at this width and budget. This is
  the more interesting hypothesis and the less likely one, and it is **unfalsifiable from one seed**.

**The next run, stated as a hypothesis first:** *retraining L=10 under a second seed will land above
L=8, i.e. H1.* The cheapest decisive test is **L=10 and L=8 at seed 1**, two cells, same budget
(~9 h and ~7 h). If both seeds put L=10 below L=8 the ladder really is non-monotone and every L\* in
this project needs re-reading; if seed 1 puts L=10 between L=8 and L=12, this cell was noise and the
bracket narrows after all -- but the measured seed spread then becomes a required error bar on every
future rung.

**Consequence for the hyper-parameter scan hshunt asked for** (does required reasoning depth track MLP
depth?): the scan's centre was going to be this cell. It cannot be, until the seed question is settled.
A scan over MLP depth whose rungs carry an unmeasured seed spread of >= 0.065 solve rate would not be
able to see the effect it is looking for.

## 10. Limitations

1. **One seed per cell, the whole point.** Stated in finding 38's notebook as limitation 1 and now the
   headline rather than a caveat.
2. **No L=9 or L=11.** The bracket (8, 12] contains four integers and two are measured.
3. **Cost.** 8 h 58 m for a cell whose interpretation now requires a second seed; the next job should
   buy seeds before it buys rungs.
4. **Fast phase only** (finding 34): every bin is above the removal floor, so "reasoning depth" means
   the log N regime.
5. **ff=1024 throughout** (finding 14): every number here is "at ff=1024".
6. **The DST fall-back** sits inside this job's wall clock; `Elapsed` (8:58:51) is real time, the
   `End - Start` difference in local time is an hour shorter. Nothing else depends on it.